**[Source]** Donghwan Project (하이퍼파라미터·디코딩·모델 선택 설계) + Jisoo Project 7·12·13단계 실행 결과(재사용 검증)
**[Status]** ADAPTED
**[Role]** 학습·평가·선택 규칙을 **결과를 보기 전에** 고정하고, 지수가 이미 실행한 pko-T5 결과를 재사용해도 되는지 검증
**[Modification]** 동환의 축소 탐색(Train 20,000·lr 2개)은 그대로 가져오지 않고, 지수의 고정 subset·동일 조건 원칙에 맞춰 재설계했다.
**이 노트북은 GPU 없이 실행된다.** 체크포인트 가중치 SHA256, 학습 update 수, 예측 파일 id·input·target 일치 여부를 실제로 계산한다.

# 07. 학습·평가 설계 고정과 기존 결과 재사용 검증

## 1. 설계 원칙
1. 데이터는 지수 FIXED(`preprocessed_final_v1`)를 그대로 쓴다. Train/Validation/Test를 다시 나누지 않는다.
2. **후보 비교는 같은 조건에서.** 동일 Train subset, 동일 seed(42), 동일 학습 조건, 동일 Validation subset, Test 미사용. subset 결과는 최종 성능이 아니다.
3. 선택된 모델만 전체 Train으로 학습한다. 이미 같은 조건으로 전체 학습된 결과가 있고 현재 FIXED 데이터와 동일함이 확인되면 **재학습하지 않고 재사용**한다(아래 셀 3).
4. Validation으로 모델·하이퍼파라미터·디코딩·checkpoint를 선택하고, Test는 13번에서 설정을 잠근 뒤 14번에서 1회만 쓴다.

## 2. 사전 고정하는 선택 규칙 (ET5 결과와 Balanced EM 결과를 보기 전에 적는다)
- **주 지표:** 전체 Validation의 어절 ROUGE-2 F1(지수 정의). 차이의 95% 신뢰구간은 **document_id 단위 bootstrap**(같은 대화의 행을 묶어 재표집)으로 구한다.
- **후보 자격(둘 다 만족):** ① 교정 필요 행 ROUGE-2가 입력 복사보다 높다 ② Balanced Exact Match(NFKC)가 0.5(입력 복사)를 넘는다.
- **우열 판정:** 주 지표 차이의 CI가 0을 포함하지 않을 때만 “유의하게 높다”고 쓴다. 포함하면 우열을 주장하지 않는다.
- **보조 지표 충돌 검사:** 주 지표의 승자가 Balanced EM에서 점추정치가 더 낮으면 “충돌”로 표시하고 자동으로 확정하지 않는다(사람이 사유를 기록).
- **디코딩:** 지수 13단계의 사전 규칙 5개(전체 R2 CI 하한>0, 교정 필요 행 R2 CI 하한>0, 과교정 ③ 증가 ≤0.5%p, 빈 출력·반복 후보가 늘지 않음, 시간 ≤5배)를 **모두** 만족할 때만 Beam을 택하고, 아니면 Greedy를 쓴다.
- **정직성 주의:** KoBART vs pko-T5 subset 비교 결과는 지수가 이미 보았다. 08번은 이를 새 지표로 다시 계산해 **일치 여부를 보고**할 뿐, 결과를 보고 선택을 바꾸지 않는다. 사전 고정의 의미가 온전한 것은 ET5 arm과 Balanced EM 충돌 검사뿐이다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 사전 고정 설정 저장 — config/experiment_config.json
CFG = {
 "dataset": {"version": MAN["dataset_version"], "sha256": MAN["sha256"], "rows": MAN["counts_final"], "official_split": "지수 document_id 기반(재분할 금지)"},
 "seed": 42,
 "common_training": {"learning_rate": 5e-5, "effective_batch_size": 16, "epochs": 1, "optimizer": "AdamW", "weight_decay": 0.01, "lr_scheduler": "linear", "warmup_ratio": 0.03,
                     "max_grad_norm": 1.0, "precision": "bf16", "padding": "dynamic (DataCollatorForSeq2Seq)", "label_pad_token_id": -100, "decoding_baseline": {"num_beams": 1, "do_sample": False}},
 "screening_subset": {"train_ids_file": "output/subset_baseline/train_subset_ids.json", "val_ids_file": "output/subset_baseline/val_subset_ids.json", "n_train": 100000, "n_val": 5000,
                      "note": "지수 8번이 만든 고정 subset. source_sha256이 현재 동결 데이터와 같은지 셀 3에서 확인"},
 "hpo_ET5": {"lr_grid": [3e-5, 5e-5], "hpo_train": "train_subset 앞 10,000행", "hpo_val": "val_subset 앞 2,000행", "criterion": "validation loss가 더 낮은 lr, 차이가 1% 미만이면 지수 고정값 5e-5 유지",
             "changed_from_donghwan": "동환은 Train 3,000/Validation 300, 1 epoch로 비교. 통합본은 지수 subset 위에서 비교하고 한계를 기록"},
 "decoding": {"compare": {"greedy": {"num_beams": 1}, "beam3": {"num_beams": 3}}, "kept_equal": {"max_new_tokens": "모델별 target max_length", "do_sample": False},
              "extra_sweep_val_subset": {"rows": "val_subset 앞 2,000행", "num_beams": [1, 2, 3, 5], "length_penalty": [0.8, 1.0, 1.2], "repetition_penalty": [1.0, 1.2],
                                         "rule": "한 번에 한 조건만 변경, Beam 전용 설정은 num_beams>1에서만", "note": "동환의 순차 탐색 참고. 결과가 근소하면 Greedy 유지"},
              "sampling": "사용하지 않음(교정은 사실성이 중요한 과제, 결과가 실행마다 달라짐)"},
 "metrics": {"primary": "어절 ROUGE-2 F1 (지수 정의, 원문 기준)", "secondary": ["Balanced EM(NFKC)", "교정 필요 EM", "원문 유지 EM", "CER", "chrF", "과교정률", "교정누락률", "어절/문자 ROUGE-1·L"],
             "bootstrap": {"unit": "document_id", "n": 2000, "seed": 42}},
 "selection_rules": {"eligibility": ["교정 필요 행 R2 > 입력 복사", "Balanced EM(NFKC) > 0.5"], "winner": "주 지표 차이 CI가 0 제외", "conflict_check": "Balanced EM 점추정치 반대면 충돌 표시(자동 확정 금지)",
                     "decoding": "지수 13단계 사전 규칙 5개 모두 만족 시 Beam"},
 "test_policy": "Test는 13번에서 config/FINAL_LOCKED를 만든 뒤 14번에서 1회만 사용. Test 결과로 모델·규칙·하이퍼파라미터를 바꾸지 않는다.",
}
(P.CONFIG / "experiment_config.json").write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", P.CONFIG / "experiment_config.json")
print(pd.Series(CFG["common_training"]).to_string())

저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/config/experiment_config.json
learning_rate                                        0.00005
effective_batch_size                                      16
epochs                                                     1
optimizer                                              AdamW
weight_decay                                            0.01
lr_scheduler                                          linear
warmup_ratio                                            0.03
max_grad_norm                                            1.0
precision                                               bf16
padding                     dynamic (DataCollatorForSeq2Seq)
label_pad_token_id                                      -100
decoding_baseline       {'num_beams': 1, 'do_sample': False}


## 3. 기존 결과 재사용 검증 (지수 pko-T5 전체 학습·디코딩 비교)
재사용 조건: 아래 항목이 **모두** 참이어야 한다. 하나라도 거짓이면 재사용하지 않고 이유를 보고한다.

In [3]:
# [셀 2] 지수의 12·13단계 실제 결과 파일 읽기
J = P.J_OUT
R12 = json.loads((J / "pkot5_full" / "pkot5_full_results.json").read_text(encoding="utf-8"))
R13 = json.loads((J / "decoding_comparison" / "decoding_comparison_results.json").read_text(encoding="utf-8"))
N13 = json.loads((J / "decoding_comparison" / "next_step_config.json").read_text(encoding="utf-8"))
CK_DIR = J / "checkpoints" / "pkot5_full_1epoch"           # 기록된 절대경로(다른 컴퓨터)는 쓰지 않고 현재 폴더 기준으로 찾는다
print("기록된 checkpoint 경로(다른 PC일 수 있음):", R13["checkpoint"]); print("현재 환경에서 찾는 경로:", CK_DIR, "| 존재:", CK_DIR.exists())
print("12단계 학습 요약:", {k: R12["train_summary"][k] for k in ("updates_done", "n_updates_planned", "final_train_loss", "precision", "gpu", "torch", "transformers", "resumed_from_checkpoint")})
print("13단계 선택:", N13["selected_decoding"], "| 사전 규칙:", {k: bool(v) for k, v in R13["criteria"].items()})

기록된 checkpoint 경로(다른 PC일 수 있음): C:\Users\KDS-18\Desktop\pro4_team3\output\checkpoints\pkot5_full_1epoch
현재 환경에서 찾는 경로: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/output/checkpoints/pkot5_full_1epoch | 존재: True
12단계 학습 요약: {'updates_done': 61670, 'n_updates_planned': 61670, 'final_train_loss': 0.1306875832490623, 'precision': 'bf16', 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU', 'torch': '2.7.1+cu128', 'transformers': '4.44.2', 'resumed_from_checkpoint': True}
13단계 선택: greedy | 사전 규칙: {'1. 전체 R2 차이 CI 하한 > 0': False, '2. 교정 필요 행 R2 차이 CI 하한 > 0': True, '3. 과교정 ③ 비율 증가 ≤ 0.5%p': False, '4. 빈 출력·반복 생성 후보가 늘지 않음': False, '5. 시간 비율 ≤ 5배(같은 세션)': True}


In [4]:
# [셀 3] 재사용 검증 — 현재 FIXED 데이터와 같은 실험인가?
import math
CHK = []
def chk(name, ok, detail=""):
    CHK.append({"항목": name, "통과": bool(ok), "상세": str(detail)}); print(("통과 " if ok else "실패 ") + name + (f" — {detail}" if detail else ""))

# 1) 가중치 SHA256 (13단계가 기록한 값과 대조)
w = CK_DIR / "model.safetensors"; t0 = time.time()
sha_now = common.sha256_file(w) if w.exists() else None
chk("checkpoint 가중치 SHA256 = 13단계 기록", sha_now == R13["checkpoint_weight_sha256"], f"{(sha_now or '없음')[:16]}… ({time.time()-t0:.0f}초)")

# 2) 학습 update 수가 '현재' Train 행 수와 맞는가 (제거 전 986,728행이었다면 61,671, 제거 후 986,718행이면 61,670)
n_now, n_before = MAN["counts_final"]["train"], MAN["counts_before"]["train"]
u_now, u_before = math.ceil(n_now / 16), math.ceil(n_before / 16)
chk("총 update 수 = ceil(현재 Train/16)", R12["train_summary"]["updates_done"] == u_now, f"기록 {R12['train_summary']['updates_done']:,} / 현재 Train 기준 {u_now:,} / 제거 전 Train 기준 {u_before:,}")
chk("update 수로 '제거 전 Train' 학습과 구분 가능", u_now != u_before, f"{u_now} vs {u_before}")

# 3) 학습 설정이 사전 고정 설정과 같은가
c, c0 = R12["train_summary"]["config"], CFG["common_training"]
same = {"lr": c["learning_rate"] == c0["learning_rate"], "effective_batch": c["effective_batch_size"] == c0["effective_batch_size"], "epochs": c["epochs"] == c0["epochs"], "optimizer": c["optimizer"] == c0["optimizer"],
        "wd": c["weight_decay"] == c0["weight_decay"], "scheduler": c["lr_scheduler"] == c0["lr_scheduler"], "warmup": c["warmup_ratio"] == c0["warmup_ratio"], "seed": c["seed"] == CFG["seed"], "precision": c["precision"] == c0["precision"]}
chk("학습 설정 = 사전 고정 설정", all(same.values()), same)

# 4) 예측 파일의 id·input·target이 현재 Validation과 같은가 (순서까지)
VAL = common.read_split(P, "validation", ["utterance_id", "document_id", "input", "target"])
vid, vin, vtg = [r["utterance_id"] for r in VAL], [r["input"] for r in VAL], [r["target"] for r in VAL]
PRED = {}
for name, p in {"12단계 Greedy": J / "pkot5_full" / "pkot5_full_val_predictions.jsonl", "13단계 Greedy": J / "decoding_comparison" / "val_predictions_greedy.jsonl", "13단계 Beam3": J / "decoding_comparison" / "val_predictions_beam3.jsonl"}.items():
    rows = common.read_jsonl(p); PRED[name] = rows
    ok = [r["utterance_id"] for r in rows] == vid and [r["input"] for r in rows] == vin and [r["target"] for r in rows] == vtg
    chk(f"{name} 예측 파일 id·input·target = 현재 Validation", ok, f"{len(rows):,}행")
same_greedy = [r["prediction"] for r in PRED["12단계 Greedy"]] == [r["prediction"] for r in PRED["13단계 Greedy"]]
print("12단계 Greedy 예측 = 13단계 Greedy 예측(같은 checkpoint·설정에서 재생성):", same_greedy)

# 5) subset id 파일이 현재 동결 데이터에서 만들어졌는가
tr_ids = {r["utterance_id"] for r in common.read_split(P, "train", ["utterance_id"], verify_sha=False)}
for nm, split_ids in (("train", tr_ids), ("validation", set(vid))):
    d = json.loads((J / "subset_baseline" / f"{'train' if nm=='train' else 'val'}_subset_ids.json").read_text(encoding="utf-8"))
    chk(f"{nm} subset source_sha256 = 현재 동결 해시", d["source_sha256"] == MAN["sha256"][f"{nm}.jsonl"], f"{len(d['utterance_ids']):,}개")
    chk(f"{nm} subset id가 현재 {nm}에 모두 존재", all(u in split_ids for u in d["utterance_ids"]))

# 6) Test 미사용 기록
chk("지수 기록: Test 사용 안 함", (R12.get("test_used") is False) and (R13.get("test_used") is False) and ("test.jsonl" not in json.dumps(R12.get("opened_files", []))), "12·13단계 opened_files/test_used")
REUSE_OK = all(x["통과"] for x in CHK)
print("\n" + "=" * 60); print("재사용 판정:", "REUSE_OK (pko-T5 전체 학습·예측 재사용 가능)" if REUSE_OK else "재사용 불가 — 실패 항목 확인"); print("=" * 60)
pd.DataFrame(CHK)

통과 checkpoint 가중치 SHA256 = 13단계 기록 — ed95a6fc64cd3902… (4초)
통과 총 update 수 = ceil(현재 Train/16) — 기록 61,670 / 현재 Train 기준 61,670 / 제거 전 Train 기준 61,671
통과 update 수로 '제거 전 Train' 학습과 구분 가능 — 61670 vs 61671
통과 학습 설정 = 사전 고정 설정 — {'lr': True, 'effective_batch': True, 'epochs': True, 'optimizer': True, 'wd': True, 'scheduler': True, 'warmup': True, 'seed': True, 'precision': True}
통과 12단계 Greedy 예측 파일 id·input·target = 현재 Validation — 54,730행
통과 13단계 Greedy 예측 파일 id·input·target = 현재 Validation — 54,730행
통과 13단계 Beam3 예측 파일 id·input·target = 현재 Validation — 54,730행
12단계 Greedy 예측 = 13단계 Greedy 예측(같은 checkpoint·설정에서 재생성): True
통과 train subset source_sha256 = 현재 동결 해시 — 100,000개
통과 train subset id가 현재 train에 모두 존재
통과 validation subset source_sha256 = 현재 동결 해시 — 5,000개
통과 validation subset id가 현재 validation에 모두 존재
통과 지수 기록: Test 사용 안 함 — 12·13단계 opened_files/test_used

재사용 판정: REUSE_OK (pko-T5 전체 학습·예측 재사용 가능)


                                                         항목  통과                                                                                                                                                      상세
0                      checkpoint 가중치 SHA256 = 13단계 기록  True                                                                                                                                   ed95a6fc64cd3902… (4초)
1                          총 update 수 = ceil(현재 Train/16)  True                                                                                          기록 61,670 / 현재 Train 기준 61,670 / 제거 전 Train 기준 61,671
2                update 수로 '제거 전 Train' 학습과 구분 가능  True                                                                                                                                            61670 vs 61671
3                                  학습 설정 = 사전 고정 설정  True  {'lr': True, 'effective_batch': True, 'epochs': True, 'optimizer': True, 'wd': True, 'schedule

In [5]:
# [셀 4] 재사용 검증 결과 저장 + 환경 기록
try:
    import torch, transformers; ENV_NOW = {"torch": torch.__version__, "transformers": transformers.__version__, "cuda": torch.cuda.is_available()}
except Exception as e:
    ENV_NOW = {"torch": None, "transformers": None, "note": f"이 실행 환경에는 torch/transformers가 없음({type(e).__name__}) — 재사용 검증은 파일 기반으로 수행"}
REC = {"reuse_ok": REUSE_OK, "checked_at": time.strftime("%Y-%m-%d %H:%M:%S"), "checks": CHK, "checkpoint_dir": str(CK_DIR), "checkpoint_weight_sha256": sha_now,
       "recorded_training_env": {k: R12["train_summary"].get(k) for k in ("torch", "transformers", "gpu", "precision")}, "current_env": ENV_NOW,
       "selected_decoding_by_jisoo_rule": N13["selected_decoding"], "test_used": False}
(P.RUNS / "reuse_verification_pkot5.json").write_text(json.dumps(REC, ensure_ascii=False, indent=2), encoding="utf-8"); print("저장:", P.RUNS / "reuse_verification_pkot5.json"); print("현재 환경:", ENV_NOW)
assert REUSE_OK, "재사용 검증 실패 → 09번에서 재학습이 필요한지 판단하기 전에 원인을 확인하세요"

저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/runs/reuse_verification_pkot5.json
현재 환경: {'torch': None, 'transformers': None, 'note': '이 실행 환경에는 torch/transformers가 없음(ModuleNotFoundError) — 재사용 검증은 파일 기반으로 수행'}


## 해석
- **재사용 판정: REUSE_OK.** 지수 13단계 기록과 체크포인트 가중치 SHA256이 일치하고, 12·13단계 예측 파일의 id·input·target이 현재 Validation과 동일하며, 학습·검증 subset 원본 해시도 현재 동결본과 같다. 학습 설정은 사전 등록값과 같다.
- **한계(과신 금지):** 가중치 SHA256이 증명하는 것은 "디코딩 단계 이후 체크포인트가 바뀌지 않았다"는 것까지다. 그 체크포인트가 현재 Train으로 학습됐다는 것은 업데이트 수(61,670 = ceil(986,718/16); 제거 전 Train이었다면 61,671)라는 **정황 증거**로만 뒷받침된다. 
- Test는 사용하지 않았다. 사전 등록된 설정(seed 42, lr 5e-5, 유효 batch 16, 1 epoch 등)과 선택 규칙은 `config/experiment_config.json`에 고정했으며, 이후 결과를 본 뒤에 바꾸지 않는다.